# Cleaning the C-MAPSS Data

Baseline cleaning: for each dataset (FD001-FD004), drop sensor / operational-setting
columns that are constant (zero variance) in the training set. A column that never
changes carries no signal for predicting RUL — it's just noise/dimensionality.

Which columns are constant differs per dataset: FD001 and FD003 run under a single
operating condition (so `op_setting_3` and several sensors are pinned), while FD002
and FD004 run under six conditions and vary more.

Cleaned files are written to `data/processed/` as CSV (with headers) using the same
base filename as the raw file, plus `_clean`:
`train_FD001.txt` -> `data/processed/train_FD001_clean.csv`.

In [1]:
from data_utils import FEATURE_COLUMNS, PROCESSED_DIR, load_all_datasets, add_train_rul

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

datasets = load_all_datasets()

for fd, d in datasets.items():
    d["train"] = add_train_rul(d["train"])

In [2]:
# Identify constant columns per dataset, using the training set,
# and drop the same columns from both train and test (they must stay aligned).
constant_columns = {}

for fd, d in datasets.items():
    train = d["train"]
    constant = [c for c in FEATURE_COLUMNS if train[c].nunique() <= 1]
    constant_columns[fd] = constant
    print(fd, "dropping constant columns:", constant)

    d["train"] = train.drop(columns=constant)
    d["test"] = d["test"].drop(columns=constant)

FD001 dropping constant columns: ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
FD002 dropping constant columns: []
FD003 dropping constant columns: ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_16', 'sensor_18', 'sensor_19']
FD004 dropping constant columns: []


In [3]:
for fd, d in datasets.items():
    d["train"].to_csv(PROCESSED_DIR / f"train_{fd}_clean.csv", index=False)
    d["test"].to_csv(PROCESSED_DIR / f"test_{fd}_clean.csv", index=False)
    d["rul"].to_csv(PROCESSED_DIR / f"RUL_{fd}_clean.csv", index=False)

print("saved cleaned datasets to", PROCESSED_DIR)

saved cleaned datasets to /Users/neomedic/Predictive_maintenance_engines/data/processed


In [4]:
datasets["FD001"]["train"].head()

,unit,cycle,op_setting_1,op_setting_2,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,21.61,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,191
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,190
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,189
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,187
